In [ ]:
import json, subprocess, sys, time, hashlib, shutil, zipfile, resource
from pathlib import Path
import torch
input_root=Path('/kaggle/input')
print('Mounted Kaggle inputs:',[p.name for p in input_root.iterdir()])
manifest_candidates=list(input_root.rglob('shared-html-reviewed-pages.jsonl'))
if not manifest_candidates: raise FileNotFoundError('shared-html-reviewed-pages.jsonl not mounted under /kaggle/input')
input_root=manifest_candidates[0].parent
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Kaggle GPU was not allocated')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','docling==2.129.0','rapidocr==3.9.2','onnxruntime==1.22.0'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','vietocr==0.3.13'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','einops==0.2.0','gdown==4.4.0'],check=True)
from importlib.metadata import version
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import LayoutObjectDetectionOptions, PdfPipelineOptions, TableFormerMode, TableStructureOptions
from docling.datamodel.stage_model_specs import ObjectDetectionModelSpec
from docling.datamodel.object_detection_engine_options import TransformersObjectDetectionEngineOptions
from docling.document_converter import DocumentConverter, ImageFormatOption
Path('/kaggle/working/tableformer_rotation.py').write_text('"""Experimental rotation of table crops before Docling\'s TableFormer inference."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Callable\nimport re\nimport subprocess\nimport tempfile\n\nimport numpy as np\nfrom PIL import Image\n\nMINIMUM_OSD_CONFIDENCE = 5.0\n\n\ndef _detect_rotation_clockwise(crop: np.ndarray) -> tuple[int, float | None]:\n    """Use Tesseract OSD; uncertain crops stay in their original orientation."""\n    with tempfile.NamedTemporaryFile(suffix=".png") as image_file:\n        Image.fromarray(crop).save(image_file.name)\n        try:\n            output = subprocess.run(\n                ["tesseract", image_file.name, "stdout", "--psm", "0", "-l", "osd"],\n                capture_output=True,\n                check=True,\n                text=True,\n            ).stdout\n        except (OSError, subprocess.CalledProcessError):\n            return 0, None\n    rotation = re.search(r"Rotate:\\s*(\\d+)", output)\n    confidence = re.search(r"Orientation confidence:\\s*([\\d.]+)", output)\n    if not rotation or not confidence:\n        return 0, None\n    confidence_value = float(confidence.group(1))\n    if confidence_value < MINIMUM_OSD_CONFIDENCE:\n        return 0, confidence_value\n    return int(rotation.group(1)) % 360, confidence_value\n\n\ndef _rotate_bbox(\n    box: dict, width: int, height: int, rotation_clockwise: int\n) -> dict:\n    """Rotate a top-left-origin bbox and preserve any extra cell metadata."""\n    left, top, right, bottom = (\n        float(box[key]) for key in ("l", "t", "r", "b")\n    )\n    corners = ((left, top), (right, top), (right, bottom), (left, bottom))\n    if rotation_clockwise == 90:\n        points = [(height - y, x) for x, y in corners]\n    elif rotation_clockwise == 180:\n        points = [(width - x, height - y) for x, y in corners]\n    elif rotation_clockwise == 270:\n        points = [(y, width - x) for x, y in corners]\n    else:\n        points = list(corners)\n    rotated = dict(box)\n    rotated.update(\n        l=min(x for x, _ in points),\n        t=min(y for _, y in points),\n        r=max(x for x, _ in points),\n        b=max(y for _, y in points),\n    )\n    return rotated\n\n\ndef enable_table_crop_rotation(\n    predictor: object, rotation_clockwise: int | str\n) -> Callable[[str], None]:\n    """Optionally orient each table crop before TableFormer and map boxes back."""\n    if rotation_clockwise != "auto" and rotation_clockwise not in {0, 90, 180, 270}:\n        raise ValueError("Table crop rotation must be auto, 0, 90, 180, or 270 degrees")\n    if rotation_clockwise == 0:\n        return lambda _page_id: None\n\n    original: Callable = predictor.multi_table_predict\n    current_page_id = {"value": None}\n    page_table_counts: dict[str, int] = {}\n    diagnostics: list[dict] = []\n    predictor.table_rotation_diagnostics = diagnostics\n\n    def rotated_multi_table_predict(\n        iocr_page: dict, table_bboxes: list[list[float]], *args, **kwargs\n    ) -> list[dict]:\n        page_image = np.asarray(iocr_page["image"])\n        page_height, page_width = page_image.shape[:2]\n        rotated_outputs = []\n\n        for table_bbox in table_bboxes:\n            x0, y0, x1, y1 = map(lambda value: int(round(value)), table_bbox)\n            x0, y0 = max(0, x0), max(0, y0)\n            x1, y1 = min(page_width, x1), min(page_height, y1)\n            if x1 <= x0 or y1 <= y0:\n                continue\n\n            crop = page_image[y0:y1, x0:x1]\n            page_id = current_page_id["value"]\n            table_index = page_table_counts.get(page_id, 0) + 1\n            page_table_counts[page_id] = table_index\n            if rotation_clockwise == "auto":\n                table_rotation, osd_confidence = _detect_rotation_clockwise(crop)\n            else:\n                table_rotation, osd_confidence = rotation_clockwise, None\n            diagnostics.append({\n                "page_id": page_id,\n                "table_index_1based": table_index,\n                "bbox_scaled": [x0, y0, x1, y1],\n                "rotation_clockwise": table_rotation,\n                "osd_confidence": osd_confidence,\n                "minimum_osd_confidence": MINIMUM_OSD_CONFIDENCE if rotation_clockwise == "auto" else None,\n            })\n            if table_rotation == 0:\n                output = original(\n                    iocr_page, [[x0, y0, x1, y1]], *args, **kwargs\n                )[0]\n                rotated_outputs.append(output)\n                continue\n\n            quarter_turns_ccw = (-table_rotation // 90) % 4\n            rotated_image = np.rot90(crop, k=quarter_turns_ccw).copy()\n            crop_height, crop_width = crop.shape[:2]\n            rotated_height, rotated_width = rotated_image.shape[:2]\n\n            rotated_tokens = []\n            for token in iocr_page.get("tokens", []):\n                cell = token.get("bbox", {})\n                left = max(float(cell.get("l", 0)), x0)\n                top = max(float(cell.get("t", 0)), y0)\n                right = min(float(cell.get("r", 0)), x1)\n                bottom = min(float(cell.get("b", 0)), y1)\n                if right <= left or bottom <= top:\n                    continue\n                local_box = dict(\n                    cell,\n                    l=left - x0,\n                    t=top - y0,\n                    r=right - x0,\n                    b=bottom - y0,\n                )\n                rotated_token = dict(token)\n                rotated_token["bbox"] = _rotate_bbox(\n                    local_box, crop_width, crop_height, table_rotation\n                )\n                rotated_tokens.append(rotated_token)\n\n            rotated_page = {\n                "image": rotated_image,\n                "width": rotated_width,\n                "height": rotated_height,\n                "tokens": rotated_tokens,\n            }\n            output = original(\n                rotated_page,\n                [[0, 0, rotated_width, rotated_height]],\n                *args,\n                **kwargs,\n            )[0]\n\n            inverse_rotation = (360 - table_rotation) % 360\n            for cell in output.get("tf_responses", []):\n                for cell_box in [cell.get("bbox"), *cell.get("text_cell_bboxes", [])]:\n                    if not cell_box:\n                        continue\n                    mapped = _rotate_bbox(\n                        cell_box,\n                        rotated_width,\n                        rotated_height,\n                        inverse_rotation,\n                    )\n                    mapped.update(\n                        l=mapped["l"] + x0,\n                        t=mapped["t"] + y0,\n                        r=mapped["r"] + x0,\n                        b=mapped["b"] + y0,\n                    )\n                    cell_box.update(mapped)\n            rotated_outputs.append(output)\n\n        return rotated_outputs\n\n    predictor.multi_table_predict = rotated_multi_table_predict\n\n    def set_page_id(page_id: str) -> None:\n        current_page_id["value"] = page_id\n\n    return set_page_id\n')
Path('/kaggle/working/docling_vietocr.py').write_text('"""Experimental Docling OCR model using RapidOCR detection and VietOCR recognition."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\nimport re\nimport subprocess\nimport tempfile\nfrom pathlib import Path\nfrom typing import ClassVar, Type\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\nfrom pydantic import Field\n\nfrom docling.datamodel.accelerator_options import AcceleratorOptions\nfrom docling.datamodel.base_models import Page\nfrom docling.datamodel.document import ConversionResult\nfrom docling.datamodel.pipeline_options import OcrOptions\nfrom docling_core.types.doc import BoundingBox, CoordOrigin\nfrom docling_core.types.doc.page import TextCell\nfrom docling.models.base_ocr_model import BaseOcrModel\nfrom docling.models.factories import get_ocr_factory\nfrom docling.models.stages.ocr.tesseract_utils import (\n    parse_tesseract_orientation,\n    tesseract_box_to_bounding_rectangle,\n)\nfrom docling.utils.profiling import TimeRecorder\n\n\nclass VietOcrOptions(OcrOptions):\n    kind: ClassVar[str] = "vietocr"\n    lang: list[str] = Field(default_factory=list)\n\n\nclass VietOcrModel(BaseOcrModel):\n    """Keep Docling\'s page regions and geometry while replacing text recognition."""\n\n    def __init__(\n        self,\n        *,\n        enabled: bool,\n        artifacts_path: Path | None,\n        options: VietOcrOptions,\n        accelerator_options: AcceleratorOptions,\n    ):\n        super().__init__(\n            enabled=enabled,\n            artifacts_path=artifacts_path,\n            options=options,\n            accelerator_options=accelerator_options,\n        )\n        self.scale = options.scale\n        import torch\n        from rapidocr import RapidOCR\n        from vietocr.tool.config import Cfg\n        from vietocr.tool.predictor import Predictor\n\n        config = Cfg.load_config_from_name("vgg_seq2seq")\n        config["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"\n        config["dataset"]["image_max_width"] = 512\n        config["predictor"]["beamsearch"] = False\n        self.recognizer = Predictor(config)\n        self.detector = RapidOCR()\n        self.orientation_diagnostics: list[dict] = []\n        self._orientation_counts: dict[str | None, int] = {}\n        self.current_reference_id: str | None = None\n\n    @staticmethod\n    def _detect_orientation(image: Image.Image) -> tuple[int, int | None, float | None]:\n        """Return correction angle and OSD evidence; uncertain crops stay unrotated."""\n        with tempfile.NamedTemporaryFile(suffix=".png") as image_file:\n            image.save(image_file.name)\n            try:\n                output = subprocess.run(\n                    ["tesseract", image_file.name, "stdout", "--psm", "0", "-l", "osd"],\n                    capture_output=True,\n                    check=True,\n                    text=True,\n                ).stdout\n            except (OSError, subprocess.CalledProcessError):\n                return 0, None, None\n        orientation = re.search(r"Orientation in degrees:\\s*(\\d+)", output)\n        confidence = re.search(r"Orientation confidence:\\s*([\\d.]+)", output)\n        orientation_degrees = int(orientation.group(1)) if orientation else None\n        correction = parse_tesseract_orientation(orientation.group(1)) if orientation else 0\n        confidence_value = float(confidence.group(1)) if confidence else None\n        return correction, orientation_degrees, confidence_value\n\n    @classmethod\n    def get_options_type(cls) -> Type[OcrOptions]:\n        return VietOcrOptions\n\n    def __call__(\n        self, conv_res: ConversionResult, page_batch: Iterable[Page]\n    ) -> Iterable[Page]:\n        if not self.enabled:\n            yield from page_batch\n            return\n\n        for page in page_batch:\n            if page._backend is None or not page._backend.is_valid():\n                yield page\n                continue\n\n            with TimeRecorder(conv_res, "ocr"):\n                ocr_rects = self.get_ocr_rects(page)\n                cells: list[TextCell] = []\n                for ocr_rect in ocr_rects:\n                    if ocr_rect.area() == 0:\n                        continue\n                    crop = page._backend.get_page_image(\n                        scale=self.scale, cropbox=ocr_rect\n                    )\n                    doc_orientation, osd_orientation, osd_confidence = self._detect_orientation(crop)\n                    region_index = self._orientation_counts.get(self.current_reference_id, 0) + 1\n                    self._orientation_counts[self.current_reference_id] = region_index\n                    self.orientation_diagnostics.append({\n                        "reference_id": self.current_reference_id,\n                        "ocr_region_index_1based": region_index,\n                        "bbox": [ocr_rect.l, ocr_rect.t, ocr_rect.r, ocr_rect.b],\n                        "crop_size": list(crop.size),\n                        "osd_orientation_degrees": osd_orientation,\n                        "correction_clockwise": doc_orientation,\n                        "osd_confidence": osd_confidence,\n                    })\n                    if doc_orientation:\n                        crop = crop.rotate(-doc_orientation, expand=True)\n                    image_size = crop.size\n                    image = cv2.cvtColor(np.asarray(crop), cv2.COLOR_RGB2BGR)\n                    detected = self.detector(\n                        image, use_det=True, use_cls=False, use_rec=False\n                    )\n                    boxes = getattr(detected, "boxes", None)\n                    if boxes is None:\n                        continue\n\n                    for index, quad in enumerate(boxes):\n                        quad = np.asarray(quad, dtype=np.float32)\n                        tl, tr, br, bl = quad\n                        width = max(\n                            int(np.linalg.norm(tr - tl)),\n                            int(np.linalg.norm(br - bl)), 1,\n                        )\n                        height = max(\n                            int(np.linalg.norm(bl - tl)),\n                            int(np.linalg.norm(br - tr)), 1,\n                        )\n                        target = np.array(\n                            [[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]],\n                            dtype=np.float32,\n                        )\n                        line = cv2.warpPerspective(\n                            image,\n                            cv2.getPerspectiveTransform(quad, target),\n                            (width, height),\n                        )\n                        text = str(\n                            self.recognizer.predict(\n                                Image.fromarray(cv2.cvtColor(line, cv2.COLOR_BGR2RGB))\n                            )\n                            or ""\n                        ).strip()\n                        if not text:\n                            continue\n                        left, top = quad.min(axis=0)\n                        right, bottom = quad.max(axis=0)\n                        box = BoundingBox(\n                            l=float(left), t=float(top), r=float(right), b=float(bottom),\n                            coord_origin=CoordOrigin.TOPLEFT,\n                        )\n                        cells.append(\n                            TextCell(\n                                index=index,\n                                text=text,\n                                orig=text,\n                                confidence=1.0,\n                                from_ocr=True,\n                                rect=tesseract_box_to_bounding_rectangle(\n                                    box,\n                                    original_offset=ocr_rect,\n                                    scale=self.scale,\n                                    orientation=doc_orientation,\n                                    im_size=image_size,\n                                ),\n                            )\n                        )\n                self.post_process_cells(cells, page, conv_res)\n            yield page\n\n\ndef register_vietocr() -> None:\n    for allow_external_plugins in (False, True):\n        factory = get_ocr_factory(allow_external_plugins=allow_external_plugins)\n        if VietOcrOptions not in factory.classes:\n            factory.register(VietOcrModel, "document-pipeline-experiment", __name__)\n')
sys.path.insert(0,'/kaggle/working')
from docling_vietocr import VietOcrOptions, register_vietocr
from tableformer_rotation import enable_table_crop_rotation
register_vietocr()
manifest_path=input_root/'shared-html-reviewed-pages.jsonl'
manifest=[json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
output_root=Path('/kaggle/working/raw_pages'); output_root.mkdir(parents=True,exist_ok=True)
options=PdfPipelineOptions(accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA,num_threads=2),do_ocr=True,do_table_structure=True,layout_options=LayoutObjectDetectionOptions(model_spec=ObjectDetectionModelSpec(name='layout_heron_101',repo_id='docling-project/docling-layout-heron-101'),engine_options=TransformersObjectDetectionEngineOptions(score_threshold=0.5)),table_structure_options=TableStructureOptions(mode=TableFormerMode.ACCURATE,do_cell_matching=True),ocr_options=VietOcrOptions())
converter=DocumentConverter(format_options={InputFormat.IMAGE:ImageFormatOption(pipeline_options=options)})
converter.initialize_pipeline(InputFormat.IMAGE)
pipeline=next(iter(converter.initialized_pipelines.values()))
set_page_id=enable_table_crop_rotation(pipeline.table_model.tf_predictor,'auto')
started=time.perf_counter(); pages=[]
for item in manifest:
    page_started=time.perf_counter(); page_id=item['reference_id']; image=input_root/item['image_file']
    try:
        set_page_id(page_id)
        converted=converter.convert(image)
        (output_root/f'{page_id}.json').write_text(json.dumps(converted.document.export_to_dict(),ensure_ascii=False)+'\n')
        pages.append({'reference_id':page_id,'status':str(converted.status),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,converted.status,flush=True)
    except Exception as exc:
        pages.append({'reference_id':page_id,'status':'failed','error':repr(exc),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,'FAILED',repr(exc),flush=True)
metadata={'dataset':'letrongnhan/vietnamese-enterprise-e2e-hard-scans','input_manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest(),'input_pages':len(manifest),'device':torch.cuda.get_device_name(0),'cuda_available':True,'docling_version':version('docling'),'ocr':'RapidOCR 3.9.2 detection + VietOCR vgg_seq2seq greedy; Tesseract OSD per OCR crop','layout_model':'docling-project/docling-layout-heron-101','layout_score_threshold':0.5,'table_cell_matching':True,'table_crop_rotation':'Tesseract OSD auto; minimum confidence 5','table_rotation_diagnostics':pipeline.table_model.tf_predictor.table_rotation_diagnostics,'table_structure':'Docling TableFormer accurate + OCR cell matching','inference_wall_seconds':round(time.perf_counter()-started,3),'successful_pages':sum(p['status']!='failed' for p in pages),'failed_pages':sum(p['status']=='failed' for p in pages),'pages':pages}
Path('/kaggle/working/kaggle-inference-metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+'\n')
shutil.make_archive('/kaggle/working/raw_pages','zip',root_dir=output_root)
print(json.dumps({k:metadata[k] for k in ('device','input_pages','successful_pages','failed_pages','inference_wall_seconds')}),flush=True)
